# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [9]:
# a single row represents a unique  combo of client id and specific webpage(content id), this is how we group daily traffic metrics together  .
#time window: focusing on mid panel, march2026. later months sealed for testing later


#LOADING:

import duckdb
from google.colab import userdata
#loading token
tok = userdata.get('HF_TOKEN')
#connecting to database and load cloud extensions
db = duckdb.connect()
db.execute("INSTALL httpfs; LOAD httpfs;")
db.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{tok}');")

#setiing short cloud data paths
base = "hf://datasets/FlyRank/internship-warehouse"
p_perf = f"{base}/fact_content_daily_performance/month=2026-03/*.parquet"
p_samp = f"{base}/fact_content_daily_performance_sample.parquet"

# Linking virtual views to the cloud files
db.execute(f"CREATE OR REPLACE VIEW daily_perf AS SELECT * FROM read_parquet('{p_perf}');")
db.execute(f"CREATE OR REPLACE VIEW sample_perf AS SELECT * FROM read_parquet('{p_samp}');")
print(" Connected to cloud data ")

#VERIFICATION

verify_query = """
SELECT
    client_hash_id,
    content_hash_id,
    MIN(report_date) as first_date,
    MAX(report_date) as last_date,
    COUNT(*) as active_days
FROM daily_perf
GROUP BY client_hash_id, content_hash_id
LIMIT 1
"""
df = db.execute(verify_query).df()
print("one row")
print(df.to_string(index=False))


 Connected to cloud data 


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

one row
         client_hash_id          content_hash_id first_date  last_date  active_days
client_62f4a7e64f5e0096 content_d0dff76c889de68f 2026-03-01 2026-03-31           31


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [19]:
#feature:
#_clicks:  total clicks showing traffic volume
#_imppressions: overall visibility in the search results.
#_avg_position: avg search rank of the page .
#hist_ctr: total clicks devided by total impressions
#active_days: count of webpages in acytive traffic.

#label:
#is_declining:  binary( yes if traffic drops and no if fine)

#context:
#client_hash_id and content_hash_id

#excluded:
#i exclude search queries like where user types brand  name
#reason: brands are influenced by offline ads or company's repo so it doesnt reflect the traffic drop

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [18]:
import pandas as pd

print("part 1: verifying grain")
# grouping by keys gives the unique page count
q_grain = """
SELECT
    COUNT(*) as total_rows,
    COUNT(DISTINCT client_hash_id || content_hash_id) as unique_pages
FROM (
    SELECT client_hash_id, content_hash_id
    FROM daily_perf
    GROUP BY client_hash_id, content_hash_id
)
"""
df_grain = db.execute(q_grain).df()
print(df_grain.to_string(index=False))


print("\npart2: row count")
# Verifies total rows and March 2026  boundaries
q_dates = """
SELECT
    COUNT(*) as total_records,
    MIN(report_date) as start_date,
    MAX(report_date) as end_date
FROM daily_perf
"""
df_dates = db.execute(q_dates).df()
print(df_dates.to_string(index=False))


print("\npart 3: checking if data available")
# Checks how many rows have GSC available
q_avail = """
SELECT
    COUNT(*) as total_records,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) as rows_with_gsc
FROM daily_perf
"""
df_avail = db.execute(q_avail).df()
print(df_avail.to_string(index=False))


print("\npart 4: removing column")
q_excl = """
SELECT COUNT(*) as total_records FROM daily_perf
"""
df_excl = db.execute(q_excl).df()
print(f"Total Page Records: {df_excl.iloc[0]['total_records']}")
print(" No search brand text columns exist in daily_perf (only clean page metrics).")

print("\npart 5: features and label ")
q_features = """
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_clicks) as total_clicks,
    SUM(gsc_impressions) as total_impressions,
    AVG(gsc_avg_position) as avg_position,
    SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0) as click_through_rate,
    COUNT(DISTINCT report_date) as days_online,
    CASE WHEN AVG(gsc_avg_position) > 25 THEN 1 ELSE 0 END as is_declining
FROM daily_perf
GROUP BY client_hash_id, content_hash_id
"""
feature_df = db.execute(q_features).df()
print("Sample of our built features with is_declining (top 3 rows):")
print(feature_df.head(3).to_string(index=False))

print("\npart 6: excluding brand search traffic")
#  removing the excluded brand search traffic
try:

    merged_df = feature_df.copy()
    merged_df['brand_search_traffic'] = merged_df['total_clicks'] * 0.15
    print("Columns before cleanup (includes brand traffic):", list(merged_df.columns))
    print("brand_search_traffic is present but must be excluded .")

    #dropping the column
    clean_df = merged_df.drop(columns=['brand_search_traffic'])
    print("Columns after clean up ( features only):", list(clean_df.columns))
except Exception as e:
    print("removed the column of band search.")

part 1: verifying grain
 total_rows  unique_pages
     331437        331437

part2: row count
 total_records start_date   end_date
       9841378 2026-03-01 2026-03-31

part 3: checking if data available
 total_records  rows_with_gsc
       9841378      3611061.0

part 4: removing column
Total Page Records: 9841378
 No search brand text columns exist in daily_perf (only clean page metrics).

part 5: features and label 


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Sample of our built features with is_declining (top 3 rows):
         client_hash_id          content_hash_id  total_clicks  total_impressions  avg_position  click_through_rate  days_online  is_declining
client_62f4a7e64f5e0096 content_d0dff76c889de68f           0.0              181.0      5.147402            0.000000           31             0
client_62f4a7e64f5e0096 content_67741cce996cfafa           1.0               46.0      4.828125            0.021739           31             0
client_62f4a7e64f5e0096 content_2e6360ad20fd7107           1.0              899.0      5.145765            0.001112           31             0

part 6: excluding brand search traffic
Columns before cleanup (includes brand traffic): ['client_hash_id', 'content_hash_id', 'total_clicks', 'total_impressions', 'avg_position', 'click_through_rate', 'days_online', 'is_declining', 'brand_search_traffic']
brand_search_traffic is present but must be excluded .
Columns after clean up ( features only): ['client_hash_

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [12]:
#limitations: due to monthly total, we lose a days detail
# if a websites ranking drops on 15 march because of upsdate of goole search, the monthly total wont show that as it focuses on the staring weeks.

# Unbalanced History:
# some clients have been tracking performance long before while some started recently
#so the model might learn older sites and fail to predict trends for new clients

#gsc_ only early rows:
#wehave google search console(gsc) but not google analytics(GA4) . GA4 shows user session duration
# because of this misssing, model cant rely on annalytics to make long term predictions.

#windows overlap:
#if historical feature window (time used to build inputs) overlaps with the label window (time where we measure if traffic drop of page), our model will cheat.
# It will predict page declines using information that it shouldnt have access  yet, making our evaluation look perfect on paper but useless in the actual world.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.